In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = ROOT / "dataset"
TRAIN = DATASET / "train"
TEST = DATASET / "test"

FILES = {
    "train_source1": TRAIN / "train_source1.tsv",
    "train_source2": TRAIN / "train_source2.tsv",
    "train_source3": TRAIN / "train_source3.tsv",
    "train_ground_truth": TRAIN / "train_ground_truth.tsv",
    "test_source1": TEST / "test_source1.tsv",
    "test_source2": TEST / "test_source2.tsv",
    "test_source3": TEST / "test_source3.tsv",
}

ARTIFACTS = ROOT / "artifacts"

con = duckdb.connect()

print("Project root:", ROOT)
print("DuckDB:", duckdb.__version__)
print("Connection ready:", con is not None)

In [ ]:
from src.normalize import (
    normalize_name,
    normalize_address,
    normalize_address_compact,
    normalize_country,
)

print("Normalization module loaded.")

In [ ]:
VALID_GT = ROOT / "notebooks" / "phase2_valid_gt.parquet"

print("Validation file exists:", VALID_GT.exists())

In [ ]:
valid_gt = con.execute(f"""
    SELECT *
    FROM read_parquet('{VALID_GT.as_posix()}')
""").fetchdf()

print("Validation S1 entities:", len(valid_gt))
display(valid_gt.head())

In [ ]:
block_gt = con.execute(f"""
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_parquet('{VALID_GT.as_posix()}')
    WHERE matched_entity_ids IS NOT NULL
      AND TRIM(matched_entity_ids) <> ''
    ORDER BY random()
    LIMIT 20000
""").fetchdf()

print("Sampled validation S1 entities:", len(block_gt))

In [ ]:
block_gt["matched_entity_id"] = (
    block_gt["matched_entity_ids"].str.split(",")
)

true_pairs = (
    block_gt[
        ["source1_entity_id", "matched_entity_id"]
    ]
    .explode("matched_entity_id")
)

true_pairs["matched_entity_id"] = (
    true_pairs["matched_entity_id"].str.strip()
)

print("True pairs:", len(true_pairs))
display(true_pairs.head())

In [ ]:
import os
import duckdb

print("CPU threads available:", os.cpu_count())

con.execute("PRAGMA threads=8")

print("DuckDB threads:", con.execute(
    "SELECT current_setting('threads')"
).fetchone())

In [ ]:
block_gt = con.execute(f"""
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_parquet('{VALID_GT.as_posix()}')
    WHERE matched_entity_ids IS NOT NULL
      AND TRIM(matched_entity_ids) <> ''
    LIMIT 1000
""").fetchdf()

print("Sample size:", len(block_gt))

In [ ]:
sample_ids = block_gt["source1_entity_id"].tolist()

sample_s1 = con.execute(f"""
    SELECT
        entity_id,
        business_name,
        country
    FROM read_csv_auto(
        '{FILES["train_source1"].as_posix()}',
        delim='\\t'
    )
    WHERE entity_id IN (
        SELECT UNNEST(?)
    )
""", [sample_ids]).fetchdf()

print("Sampled S1 rows:", len(sample_s1))

In [ ]:
display(sample_s1.head())
print("Rows:", len(sample_s1))
print("Unique IDs:", sample_s1["entity_id"].nunique())

In [ ]:
print("Sample rows:", len(sample_s1))
print("Unique IDs:", sample_s1["entity_id"].nunique())

display(sample_s1.head(10))

In [ ]:
PARQ = ARTIFACTS / "parquet"
KEYS = ARTIFACTS / "keys"
TMP  = ARTIFACTS / "duckdb_tmp"
for d in (PARQ, KEYS, TMP):
    d.mkdir(parents=True, exist_ok=True)

con.execute("PRAGMA threads=12")
con.execute("SET memory_limit='10GB'")
con.execute(f"SET temp_directory='{TMP.as_posix()}'")
con.execute("SET preserve_insertion_order=false")

def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def kp(name): return f"read_parquet('{(KEYS / f'{name}.parquet').as_posix()}')"

In [ ]:
def count_lines(p):
    with open(p, "rb") as f:
        return sum(b.count(b"\n") for b in iter(lambda: f.read(1 << 24), b""))

for name, path in FILES.items():
    out = PARQ / f"{name}.parquet"
    if not out.exists():
        con.execute(f"""
            COPY (
                SELECT * FROM read_csv('{path.as_posix()}', delim='\\t', header=true,
                                       all_varchar=true, quote='', escape='')
            ) TO '{out.as_posix()}' (FORMAT parquet, COMPRESSION zstd)
        """)
    n_pq = con.execute(f"SELECT count(*) FROM {pq(name)}").fetchone()[0]
    n_ln = count_lines(path) - 1
    print(f"{name:20s} parquet={n_pq:>12,}  lines={n_ln:>12,}  {'OK' if abs(n_pq - n_ln) <= 1 else 'MISMATCH'}")

In [ ]:
for name in FILES:
    print(name, con.execute(f"SELECT * FROM {pq(name)} LIMIT 0").fetchdf().columns.tolist())

q = " UNION ALL ".join(
    f"SELECT '{n}' AS file, country, count(*) AS n FROM {pq(n)} GROUP BY ALL"
    for n in FILES if "ground_truth" not in n
)
display(con.execute(q + " ORDER BY file, n DESC").fetchdf())

In [ ]:
con.execute(f"CREATE OR REPLACE VIEW tr_s1  AS SELECT * FROM {pq('train_source1')}")
con.execute(f"""CREATE OR REPLACE VIEW tr_tgt AS
    SELECT * FROM {pq('train_source2')} UNION ALL SELECT * FROM {pq('train_source3')}""")

con.execute(f"""
CREATE OR REPLACE TABLE gt_pairs AS
SELECT source1_entity_id AS s1_id, trim(tgt) AS tgt_id
FROM (
    SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS tgt
    FROM {pq('train_ground_truth')}
    WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> ''
)
""")

display(con.execute(f"""
WITH gt AS (SELECT * FROM {pq('train_ground_truth')})
SELECT
    (SELECT count(*) FROM gt)                                                    AS n_s1,
    (SELECT count(*) FROM gt WHERE matched_entity_ids IS NULL
                              OR trim(matched_entity_ids) = '')                  AS n_singletons,
    (SELECT count(*) FROM gt_pairs)                                              AS n_pairs,
    (SELECT count(*) FROM gt_pairs WHERE tgt_id LIKE 'S2-%')                     AS n_pairs_s2,
    (SELECT count(*) FROM gt_pairs WHERE tgt_id LIKE 'S3-%')                     AS n_pairs_s3,
    (SELECT count(*) FROM tr_s1)                                                 AS n_train_s1_rows,
    (SELECT count(*) FROM tr_tgt)                                                AS n_train_tgt_rows
""").fetchdf())

# Matches per S1 (including 0)
display(con.execute(f"""
SELECT n_matches, count(*) AS n_s1
FROM (
    SELECT g.source1_entity_id, count(p.tgt_id) AS n_matches
    FROM {pq('train_ground_truth')} g LEFT JOIN gt_pairs p ON p.s1_id = g.source1_entity_id
    GROUP BY 1
) GROUP BY 1 ORDER BY 1
""").fetchdf())

# One-to-many check: does any S2/S3 record belong to >1 S1?
display(con.execute("""
SELECT count(*) AS tgt_ids_with_multiple_s1
FROM (SELECT tgt_id FROM gt_pairs GROUP BY 1 HAVING count(DISTINCT s1_id) > 1)
""").fetchdf())

# Country agreement on true pairs
display(con.execute("""
SELECT s.country AS s1_country, t.country AS tgt_country, count(*) AS n
FROM gt_pairs p
JOIN tr_s1  s ON s.entity_id = p.s1_id
JOIN tr_tgt t ON t.entity_id = p.tgt_id
GROUP BY ALL ORDER BY n DESC
""").fetchdf())

In [ ]:
import re, time
import pyarrow as pa
import pyarrow.parquet as papq

POST_RE = re.compile(r"(?<!\d)(\d{3}\s?\d{3}|\d{5})(?!\d)")   # IN PIN (6 / 3+3), US ZIP & FR CP (5)

def extract_postcode(addr):
    if not isinstance(addr, str):
        return None
    m = POST_RE.findall(addr)
    return m[-1].replace(" ", "") if m else None

KEY_SCHEMA = pa.schema([(c, pa.string()) for c in
    ["entity_id", "country_key", "name_key", "addr_key", "addr_compact", "postcode"]])

def build_keys(name, batch=250_000):
    out = KEYS / f"{name}.parquet"
    if out.exists():
        return out
    tmp = out.with_suffix(".tmp")
    reader = con.execute(f"""
        SELECT entity_id, business_name, business_address, country FROM {pq(name)}
    """).fetch_record_batch(batch)
    writer = papq.ParquetWriter(tmp.as_posix(), KEY_SCHEMA, compression="zstd")
    try:
        for rb in reader:
            df = rb.to_pandas()
            addr = df["business_address"].map(normalize_address)
            k = pd.DataFrame({
                "entity_id":    df["entity_id"],
                "country_key":  df["country"].map(normalize_country),
                "name_key":     df["business_name"].map(normalize_name),
                "addr_key":     addr,
                "addr_compact": df["business_address"].map(normalize_address_compact),
                "postcode":     addr.map(extract_postcode),
            })
            writer.write_table(pa.Table.from_pandas(k, schema=KEY_SCHEMA, preserve_index=False))
    finally:
        writer.close()
    tmp.replace(out)
    return out

for name in FILES:
    if "ground_truth" in name:
        continue
    t0 = time.time()
    build_keys(name)
    n = con.execute(f"SELECT count(*) FROM {kp(name)}").fetchone()[0]
    print(f"{name:16s} {n:>12,} rows  {time.time() - t0:6.1f}s")

In [ ]:
N_EVAL = 20_000

con.execute(f"""
CREATE OR REPLACE TABLE eval_s1 AS
SELECT source1_entity_id AS s1_id, matched_entity_ids
FROM read_parquet('{VALID_GT.as_posix()}')
USING SAMPLE {N_EVAL} ROWS (reservoir, 42)
""")

con.execute("""
CREATE OR REPLACE TABLE eval_pairs AS
SELECT s1_id, trim(tgt) AS tgt_id
FROM (
    SELECT s1_id, unnest(string_split(matched_entity_ids, ',')) AS tgt
    FROM eval_s1
    WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> ''
)
""")

con.execute(f"""
CREATE OR REPLACE TABLE s1k AS
SELECT k.entity_id AS s1_id, k.* EXCLUDE (entity_id)
FROM {kp('train_source1')} k JOIN eval_s1 e ON k.entity_id = e.s1_id
""")

# Full train S2+S3 pool = realistic distractors
con.execute(f"""CREATE OR REPLACE VIEW tgtk AS
    SELECT * FROM {kp('train_source2')} UNION ALL SELECT * FROM {kp('train_source3')}""")

print(con.execute("""
SELECT (SELECT count(*) FROM eval_s1)  AS eval_s1,
       (SELECT count(*) FROM s1k)      AS s1_with_keys,
       (SELECT count(*) FROM eval_pairs) AS true_pairs,
       (SELECT count(*) FROM eval_s1 WHERE matched_entity_ids IS NULL
                                    OR trim(matched_entity_ids) = '') AS singletons
""").fetchdf())

In [ ]:
def eval_blocking(cand, label):
    return con.execute(f"""
    WITH c     AS (SELECT DISTINCT s1_id, tgt_id FROM {cand}),
         ntrue AS (SELECT s1_id, count(*) AS nt FROM eval_pairs GROUP BY 1),
         hits  AS (SELECT p.s1_id, count(*) AS h FROM eval_pairs p JOIN c USING (s1_id, tgt_id) GROUP BY 1),
         ncand AS (SELECT s1_id, count(*) AS nc FROM c GROUP BY 1),
         per   AS (
            SELECT e.s1_id, coalesce(nt, 0) AS nt, coalesce(h, 0) AS h, coalesce(nc, 0) AS nc
            FROM eval_s1 e
            LEFT JOIN ntrue USING (s1_id) LEFT JOIN hits USING (s1_id) LEFT JOIN ncand USING (s1_id)
         )
    SELECT '{label}'                                              AS method,
           sum(h) / sum(nt)                                       AS pair_recall,
           avg(CASE WHEN nt > 0 THEN (h > 0)::INT END)            AS s1_coverage,
           avg(CASE WHEN nt = 0 THEN 1.0
                    WHEN h  = 0 THEN 0.0
                    ELSE 1.25 * (h / nt) / (0.25 + h / nt) END)   AS ceiling_f05,
           avg(nc)                                                AS avg_cands,
           median(nc)                                             AS med_cands,
           quantile_cont(nc, 0.99)                                AS p99_cands,
           max(nc)                                                AS max_cands
    FROM per
    """).fetchdf()

In [ ]:
BLOCK_CAP = 200

for key, tbl in [("name_key", "cand_name"), ("addr_compact", "cand_addr")]:
    con.execute(f"""
    CREATE OR REPLACE TABLE {tbl} AS
    WITH big AS (
        SELECT country_key, {key} FROM tgtk WHERE {key} IS NOT NULL
        GROUP BY ALL HAVING count(*) > {BLOCK_CAP}
    )
    SELECT s.s1_id, t.entity_id AS tgt_id
    FROM s1k s
    JOIN tgtk t ON s.country_key = t.country_key AND s.{key} = t.{key}
    WHERE s.{key} IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM big b
                      WHERE b.country_key = s.country_key AND b.{key} = s.{key})
    """)

con.execute("""CREATE OR REPLACE TABLE cand_exact AS
    SELECT s1_id, tgt_id FROM cand_name UNION SELECT s1_id, tgt_id FROM cand_addr""")

display(pd.concat([
    eval_blocking("cand_name",  "exact_name"),
    eval_blocking("cand_addr",  "exact_addr"),
    eval_blocking("cand_exact", "exact_name∪addr"),
], ignore_index=True))

In [ ]:
TOK_RE = r"[\s\p{P}\p{S}]+"
DF_CAP = 1000
K_MAX  = 50

# Document frequency per (country, token) over the full target pool — streaming aggregate
con.execute(f"""
CREATE OR REPLACE TABLE tok_idf AS
WITH toks AS (
    SELECT DISTINCT entity_id, country_key, tok
    FROM (SELECT entity_id, country_key, unnest(string_split_regex(name_key, '{TOK_RE}')) AS tok
          FROM tgtk WHERE name_key IS NOT NULL)
    WHERE length(tok) >= 2
),
df AS (SELECT country_key, tok, count(*) AS df FROM toks GROUP BY ALL),
nc AS (SELECT country_key, count(*) AS n FROM tgtk GROUP BY 1)
SELECT d.country_key, d.tok, d.df, ln(nc.n / d.df) AS idf
FROM df d JOIN nc USING (country_key)
WHERE d.df <= {DF_CAP}
""")

con.execute(f"""
CREATE OR REPLACE TABLE s1_tok AS
SELECT DISTINCT s1_id, country_key, tok
FROM (SELECT s1_id, country_key, unnest(string_split_regex(name_key, '{TOK_RE}')) AS tok
      FROM s1k WHERE name_key IS NOT NULL)
WHERE length(tok) >= 2
""")

# Target postings only for tokens the eval S1 set actually uses
con.execute(f"""
CREATE OR REPLACE TABLE tgt_post AS
SELECT DISTINCT t.entity_id AS tgt_id, t.country_key, t.tok
FROM (SELECT entity_id, country_key, unnest(string_split_regex(name_key, '{TOK_RE}')) AS tok
      FROM tgtk WHERE name_key IS NOT NULL) t
SEMI JOIN (SELECT DISTINCT s.country_key, s.tok FROM s1_tok s
           JOIN tok_idf i USING (country_key, tok)) q
  ON t.country_key = q.country_key AND t.tok = q.tok
""")

con.execute(f"""
CREATE OR REPLACE TABLE cand_tok AS
WITH scored AS (
    SELECT s.s1_id, p.tgt_id, sum(i.idf) AS score, count(*) AS n_shared
    FROM s1_tok s
    JOIN tok_idf  i USING (country_key, tok)
    JOIN tgt_post p USING (country_key, tok)
    GROUP BY ALL
),
ranked AS (
    SELECT *, row_number() OVER (PARTITION BY s1_id, left(tgt_id, 2)
                                 ORDER BY score DESC, tgt_id) AS rk
    FROM scored
)
SELECT * FROM ranked WHERE rk <= {K_MAX}
""")

rows = [eval_blocking(f"(SELECT s1_id, tgt_id FROM cand_tok WHERE rk <= {k})", f"tok_top{k}/src")
        for k in (1, 3, 5, 10, 20, 50)]
rows.append(eval_blocking(
    "(SELECT s1_id, tgt_id FROM cand_tok WHERE rk <= 10 UNION SELECT s1_id, tgt_id FROM cand_exact)",
    "tok_top10 ∪ exact"))
display(pd.concat(rows, ignore_index=True))

In [ ]:
miss = con.execute("""
SELECT p.s1_id, p.tgt_id,
       s.country_key AS s1_cty,  t.country_key AS tgt_cty,
       s.name_key    AS s1_name, t.name_key    AS tgt_name,
       s.addr_key    AS s1_addr, t.addr_key    AS tgt_addr,
       s.postcode    AS s1_pc,   t.postcode    AS tgt_pc
FROM eval_pairs p
LEFT JOIN (SELECT s1_id, tgt_id FROM cand_tok WHERE rk <= 20) c USING (s1_id, tgt_id)
JOIN s1k  s USING (s1_id)
JOIN tgtk t ON t.entity_id = p.tgt_id
WHERE c.tgt_id IS NULL
ORDER BY hash(p.s1_id || p.tgt_id)
LIMIT 40
""").fetchdf()

pd.set_option("display.max_colwidth", 60)
display(miss)

In [ ]:
TOK = r"[\s\p{P}\p{S}]+"

# 1. Corpus profile per source × country
display(con.execute(r"""
WITH k AS (
    SELECT 'S1' AS src, country_key, name_key, addr_key, postcode FROM s1k
    UNION ALL
    SELECT left(entity_id, 2), country_key, name_key, addr_key, postcode FROM tgtk
)
SELECT src, country_key, count(*) AS n,
       avg(regexp_matches(coalesce(name_key,''), '[^\x00-\x7F]')::INT) AS name_nonascii,
       avg(regexp_matches(coalesce(addr_key,''), '[^\x00-\x7F]')::INT) AS addr_nonascii,
       avg((name_key IS NULL)::INT)                                     AS name_null,
       avg((addr_key IS NULL)::INT)                                     AS addr_null,
       avg((postcode IS NOT NULL)::INT)                                 AS pc_cov,
       avg(regexp_matches(coalesce(addr_key,''), '<\s*null\s*>')::INT)  AS addr_null_tag
FROM k GROUP BY ALL ORDER BY src, country_key
""").fetchdf())

# 2. What true pairs actually share -> upper bound for each blocking key type
display(con.execute(rf"""
WITH p AS (
    SELECT left(p.tgt_id, 2) AS src, s.country_key AS cty,
           s.name_key sn, t.name_key tn, s.addr_key sa, t.addr_key ta, s.postcode sp, t.postcode tp
    FROM eval_pairs p JOIN s1k s USING (s1_id) JOIN tgtk t ON t.entity_id = p.tgt_id
),
f AS (
    SELECT *,
        list_filter(string_split_regex(coalesce(sn,''), '{TOK}'), x -> length(x) >= 2) AS snt,
        list_filter(string_split_regex(coalesce(tn,''), '{TOK}'), x -> length(x) >= 2) AS tnt,
        list_filter(string_split_regex(coalesce(sa,''), '{TOK}'), x -> length(x) >= 2) AS sat,
        list_filter(string_split_regex(coalesce(ta,''), '{TOK}'), x -> length(x) >= 2) AS tat
    FROM p
)
SELECT src, cty, count(*) AS pairs,
    avg((sn = tn)::INT)                                          AS name_exact,
    avg((len(list_intersect(snt, tnt)) > 0)::INT)                AS name_share_tok,
    avg((len(list_intersect(sat, tat)) > 0)::INT)                AS addr_share_tok,
    avg((len(list_intersect(snt, tnt)) = 0
         AND len(list_intersect(sat, tat)) = 0)::INT)            AS share_nothing,
    avg(regexp_matches(coalesce(tn,''), '[^\x00-\x7F]')::INT)    AS tgt_name_nonascii,
    avg(CASE WHEN sp IS NOT NULL AND tp IS NOT NULL
             THEN (sp = tp)::INT END)                            AS pc_agree_when_both
FROM f GROUP BY ALL ORDER BY src, cty
""").fetchdf())

# 3. How many eval S1 lost every token to DF_CAP
display(con.execute("""
WITH surv AS (SELECT s1_id, count(*) AS n FROM s1_tok JOIN tok_idf USING (country_key, tok) GROUP BY 1)
SELECT avg((surv.n IS NULL)::INT) AS s1_zero_rare_tokens FROM s1k LEFT JOIN surv USING (s1_id)
""").fetchdf())

# 4. How generic are names
display(con.execute("""
SELECT country_key, name_key, count(*) AS n FROM tgtk
WHERE name_key IS NOT NULL GROUP BY ALL ORDER BY n DESC LIMIT 20
""").fetchdf())

In [ ]:
import importlib, src.normalize as N
importlib.reload(N)

for s in ["आदित्य वेंचर्स प्राइवेट लिमिटेड", "Aditya Ventures Private Limited",
          "*** Universal Digital", "Universal Digital Brands Corporation",
          "Sri Shiv Solutions Pvt Ltd", "Société Générale SA",
          "വിഷൻ ഇൻവെസ്റ്റ്മെന്റ്സ്"]:
    v2 = N.normalize_name_v2(s)
    print(f"{s!r:45} -> {v2!r:40} core={N.name_core(v2)!r}")

for a in ["0210 Linda Cir, Granville, IL", "G/F Shop Plot No -21 <null>, Chandan Place"]:
    print(repr(a), "->", repr(N.normalize_address_v2(a)))

In [ ]:
importlib.reload(N)
print(repr(N.normalize_address_v2("Boston MA 02139")))   # expect 'boston ma 02139'
print(repr(N.normalize_address_v2("27/5B Akshay Bldg"))) # expect '27 5b akshay building'
print(N._MINED, N._MINED.exists())                        # False until Cell 2 runs

In [ ]:
importlib.reload(N)
print(N._MINED.exists(), len(N.LEGAL_CANON), len(N.ADDR_CANON))
for s in ["आदित्य वेंचर्स प्राइवेट लिमिटेड", "വിഷൻ ഇൻവെസ്റ്റ്മെന്റ്സ്"]:
    v2 = N.normalize_name_v2(s); print(v2, "|", N.name_core(v2))

In [ ]:
import unicodedata
from collections import Counter, defaultdict
from anyascii import anyascii

def scripts_of(text):
    """Unicode scripts of non-ASCII letters; accented Latin shows up as LATIN."""
    if not isinstance(text, str):
        return ()
    s = set()
    for ch in text:
        if ord(ch) < 128 or not ch.isalpha():
            continue
        try:
            s.add(unicodedata.name(ch).split()[0])
        except ValueError:
            s.add("UNKNOWN")
    return tuple(sorted(s))

rows, examples = [], defaultdict(list)
for n in [f for f in FILES if "ground_truth" not in f]:
    df = con.execute(f"""
        SELECT country, business_name, business_address
        FROM {pq(n)} USING SAMPLE 200000 ROWS (reservoir, 7)
    """).fetchdf()
    for col in ("business_name", "business_address"):
        sc = df[col].map(scripts_of)
        tmp = pd.DataFrame({"country": df["country"], "script": sc, "text": df[col]})
        tmp["script"] = tmp["script"].map(lambda t: list(t) if t else ["ASCII_ONLY"])
        tmp = tmp.explode("script")
        tot = df.groupby("country").size()
        for (cty, scr), g in tmp.groupby(["country", "script"]):
            rows.append({"file": n, "field": col, "country": cty, "script": scr,
                         "n": len(g), "share": len(g) / tot[cty]})
            key = (cty, scr)
            if scr != "ASCII_ONLY" and len(examples[key]) < 3:
                examples[key].extend(g["text"].head(3 - len(examples[key])).tolist())

prof = pd.DataFrame(rows)
display(prof.pivot_table(index=["country", "script"], columns=["file", "field"],
                         values="share", aggfunc="sum").fillna(0).round(4))

print("\n=== examples + transliteration ===")
for (cty, scr), texts in sorted(examples.items()):
    for t in texts:
        print(f"[{cty:6s} {scr:12s}] {t[:60]!r:64} -> {anyascii(t)[:60]!r}")

In [ ]:
importlib.reload(N)
tests = ["Youngā\x80\x99s Cafe", "Block Â\x80\x93C, Jindal Tower", "Pacific Óptimal Freedom LLC",
         "TOURCOING, Nº 465 R DU BRUN PAIN", "Boston MA 02139", "27/5B Akshay Bldg",
         "G/F Shop Plot No -21 <null>, Chandan Place", "Sri Shiv Solutions Pvt Ltd"]
for s in tests:
    print(f"{s[:40]!r:44} addr={N.normalize_address_v2(s)!r}  name={N.normalize_name_v2(s)!r}")
print("mined file:", N._MINED.exists())

In [ ]:
con.execute(f"COPY eval_s1 TO '{(ARTIFACTS / 'eval_s1.parquet').as_posix()}' (FORMAT parquet)")

Continue 

In [1]:
import sys, re, time, importlib
from pathlib import Path
import duckdb, pandas as pd
import pyarrow as pa, pyarrow.parquet as papq

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.normalize as N

DATASET, ARTIFACTS = ROOT / "dataset", ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
KEYS2.mkdir(exist_ok=True)
FILES = {f"{s}_source{i}": None for s in ("train", "test") for i in (1, 2, 3)}

con = duckdb.connect()
con.execute("PRAGMA threads=12")
con.execute("SET memory_limit='10GB'")
con.execute(f"SET temp_directory='{(ARTIFACTS / 'duckdb_tmp').as_posix()}'")
con.execute("SET preserve_insertion_order=false")

def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con.execute(f"CREATE OR REPLACE TABLE eval_s1 AS SELECT * FROM read_parquet('{(ARTIFACTS / 'eval_s1.parquet').as_posix()}')")
con.execute("""
CREATE OR REPLACE TABLE eval_pairs AS
SELECT s1_id, trim(tgt) AS tgt_id
FROM (SELECT s1_id, unnest(string_split(matched_entity_ids, ',')) AS tgt FROM eval_s1
      WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')
""")

POST_RE = re.compile(r"(?<!\d)(\d{3}\s?\d{3}|\d{5})(?!\d)")
def extract_postcode(addr):
    if not isinstance(addr, str):
        return None
    m = POST_RE.findall(addr)
    return m[-1].replace(" ", "") if m else None

def eval_blocking(cand, label):
    return con.execute(f"""
    WITH c     AS (SELECT DISTINCT s1_id, tgt_id FROM {cand}),
         ntrue AS (SELECT s1_id, count(*) AS nt FROM eval_pairs GROUP BY 1),
         hits  AS (SELECT p.s1_id, count(*) AS h FROM eval_pairs p JOIN c USING (s1_id, tgt_id) GROUP BY 1),
         ncand AS (SELECT s1_id, count(*) AS nc FROM c GROUP BY 1),
         per   AS (SELECT e.s1_id, coalesce(nt,0) nt, coalesce(h,0) h, coalesce(nc,0) nc
                   FROM eval_s1 e LEFT JOIN ntrue USING (s1_id) LEFT JOIN hits USING (s1_id)
                   LEFT JOIN ncand USING (s1_id))
    SELECT '{label}' AS method,
           sum(h) / sum(nt)                                              AS pair_recall,
           avg(CASE WHEN nt > 0 THEN (h > 0)::INT END)                   AS s1_coverage,
           avg(CASE WHEN nt = 0 THEN 1.0 WHEN h = 0 THEN 0.0
                    ELSE 1.25 * (h / nt) / (0.25 + h / nt) END)          AS ceiling_f05,
           avg(nc) AS avg_cands, median(nc) AS med_cands,
           quantile_cont(nc, 0.99) AS p99_cands, max(nc) AS max_cands
    FROM per""").fetchdf()

print("eval:", con.execute("SELECT (SELECT count(*) FROM eval_s1), (SELECT count(*) FROM eval_pairs)").fetchone(),
      "| translit:", len(N.TRANSLIT), "| name canon:", len(N.LEGAL_CANON), "| addr canon:", len(N.ADDR_CANON))

eval: (20000, 69252) | translit: 1332 | name canon: 732 | addr canon: 1139


In [2]:
K2_COLS   = ["entity_id", "country_key", "name_v2", "name_core", "addr_v2", "postcode"]
K2_SCHEMA = pa.schema([(c, pa.string()) for c in K2_COLS])

def build_keys_v2(name, batch=250_000):
    out = KEYS2 / f"{name}.parquet"
    if out.exists():
        return
    tmp = out.with_suffix(".tmp")
    reader = con.execute(f"""
        SELECT entity_id, business_name, business_address, country FROM {pq(name)}
    """).to_arrow_reader(batch)
    writer = papq.ParquetWriter(tmp.as_posix(), K2_SCHEMA, compression="zstd")
    try:
        for rb in reader:
            df = rb.to_pandas()
            nm = df["business_name"].map(N.normalize_name_v2)
            ad = df["business_address"].map(N.normalize_address_v2)
            k = pd.DataFrame({
                "entity_id":   df["entity_id"],
                "country_key": df["country"].map(N.normalize_country_v2),
                "name_v2":     nm,
                "name_core":   nm.map(N.name_core),
                "addr_v2":     ad,
                "postcode":    ad.map(extract_postcode),
            })
            writer.write_table(pa.Table.from_pandas(k, schema=K2_SCHEMA, preserve_index=False))
    finally:
        writer.close()
    tmp.replace(out)

for name in FILES:
    t0 = time.time(); build_keys_v2(name)
    n = con.execute(f"SELECT count(*) FROM {k2(name)}").fetchone()[0]
    print(f"{name:16s} {n:>12,} rows  {time.time() - t0:7.1f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train_source1       2,206,821 rows    125.8s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AttributeError: 'float' object has no attribute 'split'

In [3]:
K2_COLS   = ["entity_id", "country_key", "name_v2", "name_core", "addr_v2", "postcode"]
K2_SCHEMA = pa.schema([(c, pa.string()) for c in K2_COLS])

def build_keys_v2(name, batch=250_000):
    out = KEYS2 / f"{name}.parquet"
    if out.exists():
        return
    tmp = out.with_suffix(".tmp")
    reader = con.execute(f"""
        SELECT entity_id, business_name, business_address, country FROM {pq(name)}
    """).to_arrow_reader(batch)
    writer = papq.ParquetWriter(tmp.as_posix(), K2_SCHEMA, compression="zstd")
    try:
        for rb in reader:
            ids   = rb.column(0).to_pylist()          # plain Python str / None
            names = rb.column(1).to_pylist()
            addrs = rb.column(2).to_pylist()
            ctys  = rb.column(3).to_pylist()

            nm = [N.normalize_name_v2(x) for x in names]
            ad = [N.normalize_address_v2(x) for x in addrs]
            cols = {
                "entity_id":   ids,
                "country_key": [N.normalize_country_v2(x) for x in ctys],
                "name_v2":     nm,
                "name_core":   [N.name_core(x) for x in nm],
                "addr_v2":     ad,
                "postcode":    [extract_postcode(x) for x in ad],
            }
            writer.write_table(pa.Table.from_pydict(cols, schema=K2_SCHEMA))
    finally:
        writer.close()
    tmp.replace(out)

for name in FILES:
    t0 = time.time(); build_keys_v2(name)
    n = con.execute(f"SELECT count(*) FROM {k2(name)}").fetchone()[0]
    print(f"{name:16s} {n:>12,} rows  {time.time() - t0:7.1f}s")

train_source1       2,206,821 rows      0.0s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train_source2       5,034,616 rows    276.1s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train_source3       5,285,603 rows    285.8s


...

test_source1        1,732,544 rows    106.0s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test_source2        4,887,273 rows    279.7s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test_source3        5,082,316 rows    284.9s


In [4]:
display(con.execute(f"""
SELECT '{'{'}f{'}'}' AS f, 1 AS dummy
""").fetchdf()) if False else None

q = " UNION ALL ".join(f"""
SELECT '{n}' AS file, count(*) AS n,
       avg((name_core IS NULL)::INT) AS name_null, avg((addr_v2 IS NULL)::INT) AS addr_null,
       avg((postcode IS NOT NULL)::INT) AS pc_cov,
       avg(regexp_matches(coalesce(name_v2,'') || coalesce(addr_v2,''), '[^\\x00-\\x7F]')::INT) AS non_ascii
FROM {k2(n)}""" for n in FILES)
display(con.execute(q).fetchdf())
display(con.execute(f"SELECT * FROM {k2('train_source2')} WHERE country_key = 'india' USING SAMPLE 8 ROWS (reservoir, 1)").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,file,n,name_null,addr_null,pc_cov,non_ascii
0,train_source1,2206821,0.000000,0.000000,0.080128,0.0
1,train_source2,5034616,0.000001,0.033561,0.094844,0.0
2,train_source3,5285603,0.000003,0.033282,0.093460,0.0
3,test_source1,1732544,0.000000,0.000000,0.056670,0.0
4,test_source2,4887273,0.000010,0.026479,0.070836,0.0
5,test_source3,5082316,0.000012,0.026779,0.069211,0.0


,entity_id,country_key,name_v2,name_core,addr_v2,postcode
0,S2-102608891,india,hitech technologies private limited,hitech technologies,vidyaagar sangamner ahmednagar maharashtra,None
1,S2-808610476,india,baba tech limited,baba tech,1 7 285 4 first floor kalasiguda m g road secu...,None


In [6]:
try:
    con.close()
except Exception:
    pass

DB = ARTIFACTS / "blocking.duckdb"
con = duckdb.connect(DB.as_posix())
con.execute("PRAGMA threads=8")
con.execute("SET memory_limit='9GB'")
con.execute(f"SET temp_directory='{(ARTIFACTS / 'duckdb_tmp').as_posix()}'")
con.execute("SET preserve_insertion_order=false")

# 'S2-335989615' -> 2*1e11 + 335989615   (source recoverable as id // 1e11)
con.execute("""CREATE OR REPLACE MACRO eid(x) AS
    (CASE left(x, 2) WHEN 'S1' THEN 1 WHEN 'S2' THEN 2 ELSE 3 END) * 100000000000
    + CAST(substr(x, 4) AS BIGINT)""")
con.execute("""CREATE OR REPLACE MACRO toks(txt) AS
    list_distinct(list_filter(string_split(coalesce(txt, ''), ' '), x -> length(x) >= 2))""")

# integer ids must be unique (guards against leading-zero collisions)
for n in FILES:
    a, b = con.execute(f"SELECT count(DISTINCT entity_id), count(DISTINCT eid(entity_id)) FROM {k2(n)}").fetchone()
    assert a == b, (n, a, b)
print("ids OK")

con.execute(f"""CREATE OR REPLACE TABLE eval_s1 AS
    SELECT eid(s1_id) AS s1_id, matched_entity_ids
    FROM read_parquet('{(ARTIFACTS / 'eval_s1.parquet').as_posix()}')""")
con.execute("""CREATE OR REPLACE TABLE eval_pairs AS
    SELECT s1_id, eid(trim(tgt)) AS tgt_id
    FROM (SELECT s1_id, unnest(string_split(matched_entity_ids, ',')) AS tgt FROM eval_s1
          WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> '')""")
print(con.execute("SELECT (SELECT count(*) FROM eval_s1), (SELECT count(*) FROM eval_pairs)").fetchone())

ids OK
(20000, 69252)


In [7]:
NAME_TOP, ADDR_TOP, KEY_CAP, K_MAX = 3, 3, 300, 50
SRC = 100000000000

TGT = f"(SELECT * FROM {k2('train_source2')} UNION ALL SELECT * FROM {k2('train_source3')})"
S1  = f"(SELECT k.* FROM {k2('train_source1')} k SEMI JOIN eval_s1 e ON eid(k.entity_id) = e.s1_id)"

def tok_sql(src, c):
    return f"""
    SELECT eid(entity_id) AS id, 'n' AS f, hash(tok) AS h
    FROM (SELECT entity_id, unnest(toks(name_core)) AS tok FROM {src} WHERE country_key = '{c}')
    UNION ALL
    SELECT eid(entity_id), 'a', hash(tok)
    FROM (SELECT entity_id, unnest(toks(addr_v2)) AS tok FROM {src} WHERE country_key = '{c}')"""

def keys_sql(src, tok_tbl, c):
    return f"""
    WITH sel AS (
        SELECT id, f, h FROM (
            SELECT t.id, t.f, t.h,
                   row_number() OVER (PARTITION BY t.id, t.f ORDER BY d.df, t.h) AS r
            FROM {tok_tbl} t JOIN df_c d USING (f, h)
        ) WHERE (f = 'n' AND r <= {NAME_TOP}) OR (f = 'a' AND r <= {ADDR_TOP})
    )
    SELECT x.id, hash(concat_ws('|', x.f || y.f, x.h::VARCHAR, y.h::VARCHAR)) AS k
    FROM sel x JOIN sel y USING (id)
    WHERE (x.f = 'n' AND y.f = 'a') OR (x.f = y.f AND x.h < y.h)
    UNION ALL
    SELECT eid(entity_id), hash(concat_ws('|', 'core', name_core))
    FROM {src} WHERE country_key = '{c}' AND name_core IS NOT NULL"""

con.execute("CREATE OR REPLACE TABLE cand2 (s1_id BIGINT, tgt_id BIGINT, score DOUBLE, n_keys BIGINT, rk BIGINT)")
countries = [r[0] for r in con.execute(f"SELECT DISTINCT country_key FROM {S1}").fetchall()]

t0 = time.time()
for c in countries:
    con.execute(f"CREATE OR REPLACE TABLE t_tok AS {tok_sql(TGT, c)}")
    con.execute("CREATE OR REPLACE TABLE df_c AS SELECT f, h, count(*) AS df FROM t_tok GROUP BY ALL")
    con.execute(f"CREATE OR REPLACE TABLE t_keys AS {keys_sql(TGT, 't_tok', c)}")
    con.execute(f"""CREATE OR REPLACE TABLE key_w AS
        SELECT k, count(*) AS df FROM t_keys GROUP BY 1 HAVING count(*) <= {KEY_CAP}""")
    con.execute(f"CREATE OR REPLACE TABLE s_tok AS {tok_sql(S1, c)}")
    con.execute(f"CREATE OR REPLACE TABLE s_keys AS {keys_sql(S1, 's_tok', c)}")
    con.execute(f"""
    INSERT INTO cand2
    SELECT * FROM (
        SELECT s1_id, tgt_id, score, n_keys,
               row_number() OVER (PARTITION BY s1_id, tgt_id // {SRC} ORDER BY score DESC, tgt_id) AS rk
        FROM (
            SELECT s.id AS s1_id, t.id AS tgt_id, sum(ln(1e7 / w.df)) AS score, count(*) AS n_keys
            FROM s_keys s
            JOIN key_w  w USING (k)
            JOIN t_keys t USING (k)
            GROUP BY ALL
        )
    ) WHERE rk <= {K_MAX}
    """)
    nk = con.execute("SELECT count(*) FROM t_keys").fetchone()[0]
    print(f"{c:8s} target keys={nk:>13,}  elapsed={time.time() - t0:6.0f}s")

for t in ("t_tok", "df_c", "t_keys", "key_w", "s_tok", "s_keys"):
    con.execute(f"DROP TABLE IF EXISTS {t}")
con.execute("CHECKPOINT")

print(con.execute("""
SELECT avg((c.s1_id IS NULL)::INT) AS s1_zero_cands
FROM eval_s1 e LEFT JOIN (SELECT DISTINCT s1_id FROM cand2) c USING (s1_id)
""").fetchdf())

display(pd.concat([eval_blocking(f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"v2_top{k}/src")
                   for k in (1, 3, 5, 10, 20, 50)], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: Allocation failure

In [11]:
NAME_TOP, ADDR_TOP, KEY_CAP, K_MAX = 3, 3, 300, 50
SRC, N_CHUNKS = 100000000000, 16
con.execute("PRAGMA threads=4")
con.execute("SET memory_limit='4GB'")
con.execute("SET preserve_insertion_order=false")

TGT_PARTS = [k2("train_source2"), k2("train_source3")]
TGT = f"(SELECT * FROM {TGT_PARTS[0]} UNION ALL SELECT * FROM {TGT_PARTS[1]})"
S1  = f"(SELECT k.* FROM {k2('train_source1')} k SEMI JOIN eval_s1 e ON eid(k.entity_id) = e.s1_id)"


def tok_stream(src, c):
    """One row per (record, field, token): id, f ('n' name / 'a' address), h (token hash)."""
    return f"""
    SELECT eid(entity_id) AS id, 'n' AS f, hash(tok) AS h
    FROM (SELECT entity_id, unnest(toks(name_core)) AS tok FROM {src} WHERE country_key = '{c}')
    UNION ALL
    SELECT eid(entity_id), 'a', hash(tok)
    FROM (SELECT entity_id, unnest(toks(addr_v2)) AS tok FROM {src} WHERE country_key = '{c}')"""


def rec_sql(src, c):
    """Per record: its NAME_TOP rarest name tokens (ns) and ADDR_TOP rarest address tokens (ad)."""
    return f"""
    SELECT t.id,
           arg_min(t.h, (d.df::HUGEINT << 64) + t.h::HUGEINT, {NAME_TOP}) FILTER (WHERE t.f = 'n') AS ns,
           arg_min(t.h, (d.df::HUGEINT << 64) + t.h::HUGEINT, {ADDR_TOP}) FILTER (WHERE t.f = 'a') AS ad
    FROM ({tok_stream(src, c)}) t
    JOIN df_c d ON t.f = d.f AND t.h = d.h
    GROUP BY t.id"""


def pair_sql(rec_tbl, tag, l1, l2, cond):
    """Combined keys from two token lists of the same record."""
    return f"""
    SELECT id, hash(concat_ws('|', '{tag}', x::VARCHAR, y::VARCHAR)) AS k
    FROM (SELECT id, x, unnest(l2) AS y
          FROM (SELECT id, unnest({l1}) AS x, {l2} AS l2 FROM {rec_tbl}
                WHERE {l1} IS NOT NULL AND {l2} IS NOT NULL))
    WHERE {cond}"""


def keys_sql(src, rec_tbl, c):
    """name x addr, name x name, addr x addr, + exact name_core fallback."""
    return " UNION ALL ".join([
        pair_sql(rec_tbl, "na", "ns", "ad", "TRUE"),
        pair_sql(rec_tbl, "nn", "ns", "ns", "x < y"),
        pair_sql(rec_tbl, "aa", "ad", "ad", "x < y"),
        f"""SELECT eid(entity_id) AS id, hash(concat_ws('|', 'core', name_core)) AS k
            FROM {src} WHERE country_key = '{c}' AND name_core IS NOT NULL""",
    ])


def step(msg, t0):
    print(f"   {msg:<48} {time.time() - t0:7.0f}s", flush=True)


con.execute("CREATE OR REPLACE TABLE cand2 (s1_id BIGINT, tgt_id BIGINT, score DOUBLE, n_keys BIGINT, rk BIGINT)")
countries = [r[0] for r in con.execute(f"SELECT DISTINCT country_key FROM {S1}").fetchall()]
print("countries:", countries)

t0 = time.time()
for c in countries:
    print(f"== {c}")

    # 1. token document frequency over the full target pool (streaming aggregate)
    con.execute(f"""CREATE OR REPLACE TABLE df_c AS
        SELECT f, h, count(*) AS df FROM ({tok_stream(TGT, c)}) GROUP BY ALL""")
    step("token df", t0)

    # 2. S1 side (small)
    con.execute(f"CREATE OR REPLACE TABLE s_rec AS {rec_sql(S1, c)}")
    con.execute(f"CREATE OR REPLACE TABLE s_keys AS SELECT DISTINCT * FROM ({keys_sql(S1, 's_rec', c)})")
    con.execute("CREATE OR REPLACE TABLE s_kset AS SELECT DISTINCT k FROM s_keys")
    step("S1 keys", t0)

    # 3. target side: per source, in N_CHUNKS id-hash chunks (keeps the arg_min aggregate small)
    con.execute("CREATE OR REPLACE TABLE t_keys (id BIGINT, k UBIGINT)")
    for part in TGT_PARTS:
        for i in range(N_CHUNKS):
            chunk = f"(SELECT * FROM {part} WHERE hash(entity_id) % {N_CHUNKS} = {i})"
            con.execute(f"CREATE OR REPLACE TABLE t_rec AS {rec_sql(chunk, c)}")
            con.execute(f"""
                INSERT INTO t_keys
                SELECT t.id, t.k FROM ({keys_sql(chunk, 't_rec', c)}) t
                SEMI JOIN s_kset s ON t.k = s.k""")
            step(f"{part[-24:-2]} chunk {i + 1}/{N_CHUNKS}", t0)

    # 4. key weights (block-size cap) and candidate scoring
    con.execute(f"""CREATE OR REPLACE TABLE key_w AS
        SELECT k, count(*) AS df FROM t_keys GROUP BY 1 HAVING count(*) <= {KEY_CAP}""")
    con.execute(f"""
    INSERT INTO cand2
    SELECT * FROM (
        SELECT s1_id, tgt_id, score, n_keys,
               row_number() OVER (PARTITION BY s1_id, tgt_id // {SRC} ORDER BY score DESC, tgt_id) AS rk
        FROM (
            SELECT s.id AS s1_id, t.id AS tgt_id, sum(ln(1e7 / w.df)) AS score, count(*) AS n_keys
            FROM s_keys s
            JOIN key_w  w USING (k)
            JOIN t_keys t USING (k)
            GROUP BY ALL
        )
    ) WHERE rk <= {K_MAX}
    """)
    nk = con.execute("SELECT count(*) FROM t_keys").fetchone()[0]
    step(f"candidates (kept target keys {nk:,})", t0)

for t in ("df_c", "s_rec", "s_keys", "s_kset", "t_rec", "t_keys", "key_w"):
    con.execute(f"DROP TABLE IF EXISTS {t}")
con.execute("CHECKPOINT")

print(con.execute("""
SELECT avg((c.s1_id IS NULL)::INT) AS s1_zero_cands
FROM eval_s1 e LEFT JOIN (SELECT DISTINCT s1_id FROM cand2) c USING (s1_id)
""").fetchdf())

display(pd.concat([eval_blocking(f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"v2_top{k}/src")
                   for k in (1, 3, 5, 10, 20, 50)], ignore_index=True))

countries: ['india', 'us']
== india


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   token df                                               7s
   S1 keys                                                8s
   /train_source2.parquet chunk 1/16                     10s
   /train_source2.parquet chunk 2/16                     12s
   /train_source2.parquet chunk 3/16                     14s
   /train_source2.parquet chunk 4/16                     16s
   /train_source2.parquet chunk 5/16                     18s
   /train_source2.parquet chunk 6/16                     20s
   /train_source2.parquet chunk 7/16                     22s
   /train_source2.parquet chunk 8/16                     24s
   /train_source2.parquet chunk 9/16                     26s
   /train_source2.parquet chunk 10/16                    28s
   /train_source2.parquet chunk 11/16                    30s
   /train_source2.parquet chunk 12/16                    32s
   /train_source2.parquet chunk 13/16                    33s
   /train_source2.parquet chunk 14/16                    35s
   /train_source2.parque

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   token df                                              80s
   S1 keys                                               80s
   /train_source2.parquet chunk 1/16                     82s
   /train_source2.parquet chunk 2/16                     85s
   /train_source2.parquet chunk 3/16                     87s
   /train_source2.parquet chunk 4/16                     89s
   /train_source2.parquet chunk 5/16                     91s
   /train_source2.parquet chunk 6/16                     93s
   /train_source2.parquet chunk 7/16                     95s
   /train_source2.parquet chunk 8/16                     97s
   /train_source2.parquet chunk 9/16                     99s
   /train_source2.parquet chunk 10/16                   101s
   /train_source2.parquet chunk 11/16                   103s
   /train_source2.parquet chunk 12/16                   105s
   /train_source2.parquet chunk 13/16                   107s
   /train_source2.parquet chunk 14/16                   109s
   /train_source2.parque

,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,v2_top1/src,0.445402,0.974750,0.800794,1.99550,2.0,2.0,2
1,v2_top3/src,0.856279,0.992922,0.961950,5.91115,6.0,6.0,6
2,v2_top5/src,0.927872,0.993925,0.976697,9.67455,10.0,10.0,10
3,v2_top10/src,0.947684,0.995087,0.982295,18.46915,20.0,20.0,20
4,v2_top20/src,0.956492,0.995510,0.984882,33.94220,40.0,40.0,40
5,v2_top50/src,0.965200,0.996831,0.988173,70.18485,99.5,100.0,100


In [12]:
S1CTY = f"(SELECT eid(entity_id) AS s1_id, country_key FROM {k2('train_source1')})"
display(con.execute(f"""
WITH p AS (SELECT p.*, k.country_key, p.tgt_id // {SRC} AS src
           FROM eval_pairs p JOIN {S1CTY} k USING (s1_id))
SELECT country_key, src, count(*) AS pairs,
       avg((c5.tgt_id  IS NOT NULL)::INT) AS recall_top5,
       avg((c20.tgt_id IS NOT NULL)::INT) AS recall_top20,
       avg((c50.tgt_id IS NOT NULL)::INT) AS recall_top50
FROM p
LEFT JOIN (SELECT s1_id, tgt_id FROM cand2 WHERE rk <= 5)  c5  USING (s1_id, tgt_id)
LEFT JOIN (SELECT s1_id, tgt_id FROM cand2 WHERE rk <= 20) c20 USING (s1_id, tgt_id)
LEFT JOIN (SELECT s1_id, tgt_id FROM cand2 WHERE rk <= 50) c50 USING (s1_id, tgt_id)
GROUP BY ALL ORDER BY ALL
""").fetchdf())

,country_key,src,pairs,recall_top5,recall_top20,recall_top50
0,india,2,13379,0.932431,0.952014,0.961656
1,india,3,14262,0.879400,0.916071,0.928832
2,us,2,20081,0.938897,0.968876,0.976943
3,us,3,21530,0.946865,0.974501,0.980539


In [13]:
miss = con.execute(f"""
WITH m AS (
    SELECT p.s1_id, p.tgt_id, c.rk
    FROM eval_pairs p
    LEFT JOIN cand2 c USING (s1_id, tgt_id)
    WHERE c.rk IS NULL OR c.rk > 5
),
s AS (SELECT * FROM m ORDER BY hash(s1_id + tgt_id) LIMIT 40)
SELECT s.rk, k1.country_key AS cty,
       k1.name_core AS s1_core, kt.name_core AS tgt_core,
       k1.addr_v2   AS s1_addr, kt.addr_v2   AS tgt_addr
FROM s
JOIN (SELECT eid(entity_id) AS id, * FROM {k2('train_source1')}) k1 ON k1.id = s.s1_id
JOIN (SELECT eid(entity_id) AS id, * FROM {TGT}) kt ON kt.id = s.tgt_id
ORDER BY s.rk NULLS LAST
""").fetchdf()

print(con.execute("""
SELECT CASE WHEN c.rk IS NULL THEN 'not retrieved (>50)' ELSE 'ranked 6-50' END AS kind, count(*) AS n
FROM eval_pairs p LEFT JOIN cand2 c USING (s1_id, tgt_id)
WHERE c.rk IS NULL OR c.rk > 5 GROUP BY 1
""").fetchdf())
pd.set_option("display.max_colwidth", 55)
display(miss)

                  kind     n
0  not retrieved (>50)  2410
1          ranked 6-50  2585


,rk,cty,s1_core,tgt_core,s1_addr,tgt_addr
0,6,us,billings regional highland association,synzeta,3211 driftwood lane billings mount,mount billings 03211 driftwood lane
1,6,us,clyial systems,clyialsystems com,546 torchwood road carthage nc,torchwood road carthage nc
2,6,india,indore india,indore india,122 sarv sampanna nagar padam main regency indore m...,NaN
3,6,india,jay trading,jaitrading,w 123 first floor greater kailash 1 delhi south del...,w c 123 first floor greater kailash 1 delhi delhi
4,6,india,zodiac foods,zodiac foods,shop no 353 3rd floor a rue mall utran chorasi sura...,NaN
5,7,us,ferron boston,ferronboston,62 hillside avenue ma melrose,hillside avenue melrose ma
6,7,india,vadodara life,vadodara life,shop no tf 69 siddheshwar hallmark ajwa road vadoda...,sdop no tf 69 vadodara gujarat
7,8,india,urban white energy,urban white energy,sb 89 sector 63a noida gautam buddha nagar uttar pr...,NaN
8,9,us,robinson highland procap,robinsonhighland,2977 treadwell lane fairfax county va,2977 treadwell lane herndon va
9,9,us,jwl ace berkley,jwl ace bgrkly,4630 gamble road cedar hill tn,NaN


In [14]:
importlib.reload(N)
for s in ["Clyial Systems", "clyialsystems.com", "www.OncologyAce.com", "Dynamics Photonics L L C",
          "The New Delhi Auto", "Mr WinnerTelecom.com", "6reat Guild", "Shree Enterprises", "JWL Ace Berkley"]:
    print(f"{s!r:32} -> {N.normalize_name_v2(s)!r:36} core={N.name_core(N.normalize_name_v2(s))!r}")
for a in ["03211 Driftwood Lane", "3211 driftwood lane", "A 8 C New Delhi"]:
    print(f"{a!r:32} -> {N.normalize_address_v2(a)!r}")

'Clyial Systems'                 -> 'clyial systems'                     core='clyial systems'
'clyialsystems.com'              -> 'clyialsystems'                      core='clyialsystems'
'www.OncologyAce.com'            -> 'oncologyace'                        core='oncologyace'
'Dynamics Photonics L L C'       -> 'dynamics photonics llc'             core='dynamics photonics'
'The New Delhi Auto'             -> 'the new delhi auto'                 core='new delhi auto'
'Mr WinnerTelecom.com'           -> 'mr winnertelecom'                   core='winnertelecom'
'6reat Guild'                    -> 'great guild'                        core='great guild'
'Shree Enterprises'              -> 'shree enterprises'                  core='enterprises'
'JWL Ace Berkley'                -> 'jwl ace berkley'                    core='jwl ace berkley'
'03211 Driftwood Lane'           -> '3211 driftwood lane'
'3211 driftwood lane'            -> '3211 driftwood lane'
'A 8 C New Delhi'                ->

In [15]:
NAME_TOP, ADDR_TOP, KEY_CAP, K_MAX = 3, 4, 300, 50
SRC, N_CHUNKS = 100000000000, 16
con.execute("PRAGMA threads=4")
con.execute("SET memory_limit='4GB'")
con.execute("SET preserve_insertion_order=false")

# keep tokens of length >= 2, plus any pure number (house numbers like '8')
con.execute("""CREATE OR REPLACE MACRO toks(txt) AS
    list_distinct(list_filter(string_split(coalesce(txt, ''), ' '),
                              x -> length(x) >= 2 OR regexp_full_match(x, '[0-9]+')))""")

TGT_PARTS = [k2("train_source2"), k2("train_source3")]
TGT   = f"(SELECT * FROM {TGT_PARTS[0]} UNION ALL SELECT * FROM {TGT_PARTS[1]})"
S1ALL = k2("train_source1")                                   # full S1 pool -> vocabulary
S1    = f"(SELECT k.* FROM {S1ALL} k SEMI JOIN eval_s1 e ON eid(k.entity_id) = e.s1_id)"


def tok_stream(src, c):
    return f"""
    SELECT eid(entity_id) AS id, 'n' AS f, hash(tok) AS h
    FROM (SELECT entity_id, unnest(toks(name_core)) AS tok FROM {src} WHERE country_key = '{c}')
    UNION ALL
    SELECT eid(entity_id), 'a', hash(tok)
    FROM (SELECT entity_id, unnest(toks(addr_v2)) AS tok FROM {src} WHERE country_key = '{c}')"""


def rec_sql(src, c):
    return f"""
    SELECT t.id,
           arg_min(t.h, (d.df::HUGEINT << 64) + t.h::HUGEINT, {NAME_TOP}) FILTER (WHERE t.f = 'n') AS ns,
           arg_min(t.h, (d.df::HUGEINT << 64) + t.h::HUGEINT, {ADDR_TOP}) FILTER (WHERE t.f = 'a') AS ad
    FROM ({tok_stream(src, c)}) t
    JOIN df_c d ON t.f = d.f AND t.h = d.h
    GROUP BY t.id"""


def pair_sql(rec_tbl, tag, l1, l2, cond):
    return f"""
    SELECT id, hash(concat_ws('|', '{tag}', x::VARCHAR, y::VARCHAR)) AS k
    FROM (SELECT id, x, unnest(l2) AS y
          FROM (SELECT id, unnest({l1}) AS x, {l2} AS l2 FROM {rec_tbl}
                WHERE {l1} IS NOT NULL AND {l2} IS NOT NULL))
    WHERE {cond}"""


def keys_sql(src, rec_tbl, c):
    return " UNION ALL ".join([
        pair_sql(rec_tbl, "na", "ns", "ad", "TRUE"),
        pair_sql(rec_tbl, "nn", "ns", "ns", "x < y"),
        pair_sql(rec_tbl, "aa", "ad", "ad", "x < y"),
        f"""SELECT eid(entity_id) AS id, hash(concat_ws('|', 'core', name_core)) AS k
            FROM {src} WHERE country_key = '{c}' AND name_core IS NOT NULL""",
        f"""SELECT eid(entity_id) AS id, hash(concat_ws('|', 'cmp', replace(name_core, ' ', ''))) AS k
            FROM {src} WHERE country_key = '{c}' AND name_core IS NOT NULL""",
    ])


def step(msg, t0):
    print(f"   {msg:<48} {time.time() - t0:7.0f}s", flush=True)


con.execute("CREATE OR REPLACE TABLE cand2 (s1_id BIGINT, tgt_id BIGINT, score DOUBLE, n_keys BIGINT, rk BIGINT)")
countries = [r[0] for r in con.execute(f"SELECT DISTINCT country_key FROM {S1}").fetchall()]
print("countries:", countries)

t0 = time.time()
for c in countries:
    print(f"== {c}")

    # 1. token df over targets, restricted to tokens that also exist in the S1 vocabulary
    con.execute(f"CREATE OR REPLACE TABLE s1_vocab AS SELECT DISTINCT f, h FROM ({tok_stream(S1ALL, c)})")
    con.execute(f"""CREATE OR REPLACE TABLE df_c AS
        SELECT t.f, t.h, count(*) AS df FROM ({tok_stream(TGT, c)}) t
        SEMI JOIN s1_vocab v ON t.f = v.f AND t.h = v.h
        GROUP BY ALL""")
    step("token df (S1-vocab filtered)", t0)

    # 2. S1 side
    con.execute(f"CREATE OR REPLACE TABLE s_rec AS {rec_sql(S1, c)}")
    con.execute(f"CREATE OR REPLACE TABLE s_keys AS SELECT DISTINCT * FROM ({keys_sql(S1, 's_rec', c)})")
    con.execute("CREATE OR REPLACE TABLE s_kset AS SELECT DISTINCT k FROM s_keys")
    step("S1 keys", t0)

    # 3. target side in chunks
    con.execute("CREATE OR REPLACE TABLE t_keys (id BIGINT, k UBIGINT)")
    for part in TGT_PARTS:
        for i in range(N_CHUNKS):
            chunk = f"(SELECT * FROM {part} WHERE hash(entity_id) % {N_CHUNKS} = {i})"
            con.execute(f"CREATE OR REPLACE TABLE t_rec AS {rec_sql(chunk, c)}")
            con.execute(f"""
                INSERT INTO t_keys
                SELECT t.id, t.k FROM ({keys_sql(chunk, 't_rec', c)}) t
                SEMI JOIN s_kset s ON t.k = s.k""")
        step(f"target keys {part[-24:-2]}", t0)

    # 4. weights + candidates
    con.execute(f"""CREATE OR REPLACE TABLE key_w AS
        SELECT k, count(*) AS df FROM t_keys GROUP BY 1 HAVING count(*) <= {KEY_CAP}""")
    con.execute(f"""
    INSERT INTO cand2
    SELECT * FROM (
        SELECT s1_id, tgt_id, score, n_keys,
               row_number() OVER (PARTITION BY s1_id, tgt_id // {SRC} ORDER BY score DESC, tgt_id) AS rk
        FROM (
            SELECT s.id AS s1_id, t.id AS tgt_id, sum(ln(1e7 / w.df)) AS score, count(*) AS n_keys
            FROM s_keys s
            JOIN key_w  w USING (k)
            JOIN t_keys t USING (k)
            GROUP BY ALL
        )
    ) WHERE rk <= {K_MAX}
    """)
    nk = con.execute("SELECT count(*) FROM t_keys").fetchone()[0]
    step(f"candidates (kept target keys {nk:,})", t0)

for t in ("s1_vocab", "df_c", "s_rec", "s_keys", "s_kset", "t_rec", "t_keys", "key_w"):
    con.execute(f"DROP TABLE IF EXISTS {t}")
con.execute("CHECKPOINT")

display(pd.concat([eval_blocking(f"(SELECT s1_id, tgt_id FROM cand2 WHERE rk <= {k})", f"v3_top{k}/src")
                   for k in (1, 3, 5, 10, 20, 50)], ignore_index=True))

countries: ['us', 'india']
== us


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   token df (S1-vocab filtered)                          11s
   S1 keys                                               12s
   target keys /train_source2.parquet                    54s
   target keys /train_source3.parquet                    98s
   candidates (kept target keys 12,016,163)             100s
== india


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   token df (S1-vocab filtered)                         111s
   S1 keys                                              112s
   target keys /train_source2.parquet                   151s
   target keys /train_source3.parquet                   192s
   candidates (kept target keys 5,768,705)              193s


,method,pair_recall,s1_coverage,ceiling_f05,avg_cands,med_cands,p99_cands,max_cands
0,v3_top1/src,0.450745,0.977656,0.806126,1.99885,2.0,2.0,2
1,v3_top3/src,0.867181,0.994823,0.966831,5.97940,6.0,6.0,6
2,v3_top5/src,0.939756,0.995563,0.981328,9.92060,10.0,10.0,10
3,v3_top10/src,0.956536,0.996514,0.985936,19.61145,20.0,20.0,20
4,v3_top20/src,0.964694,0.997200,0.988487,38.30785,40.0,40.0,40
5,v3_top50/src,0.972896,0.997887,0.991302,89.32390,100.0,100.0,100
